# Maps of uncertainty by source 

Both raw and normalized

In [2]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import string
import zarr
import warnings
from tqdm.notebook import tqdm

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
from cartopy import crs as ccrs
import cmocean

import distributed

from funcs_support import get_filepaths,get_params,utility_save,utility_print,load_shp
from funcs_readregs import load_dmgfs
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle, plot_dmgfs

dir_list = get_params()
extra_fonts = {'lato':mpl.font_manager.FontProperties(fname=dir_list['aux']+'fonts/Lato-Regular.ttf'),
 'lato-bold':mpl.font_manager.FontProperties(fname=dir_list['aux']+'fonts/Lato-Bold.ttf')}


In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','shp_fn':'impact-region','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','shp_fn':'country-fixed','popwght':True},
               'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','shp_fn':'UScounties','popwght':True}}

proc_names = {'mortality':'mort. rate',
              'gdppc':'GDP p.c. growth',
              'payroll':'payroll'}

proc_full_names = {'mortality':'All-cause mortality\n(Carleton et al. 2022)',
                   'gdppc':'GDP per capita growth\n(Burke, Hsiang, and\nMiguel 2015)',
                   'payroll':'Payroll\n(Behrer et al., 2021)'}

idxs = {'impact-regions':{'hierid':'locid'},
        'adm1':{'CNTRY_NAME':'locid'},
        'adm2-us':{'FIPS':'locid'}}




In [ ]:
save_fig = True
for gwl in [1.5,2,3]:
    for typ in ['raw','norm']:
        #typ = 'raw'
        output_fn = f'{dir_list['figs']}extra_figs/uncert_source_maps_gwl{str(gwl)}_{typ}'
        
        imp_params = {'mortality':{'timeframe':'1980-2010','obstreat':'sep','resolution':'025deg','spec':'pooled','indflag':'noInd'},
                      'gdppc':{'timeframe':'1980-2010','obstreat':'sep','resolution':'025deg'},
                      'payroll':{'timeframe':'1986-2008','obstreat':'sep','resolution':'025deg','weighting':'pop'}}
        
        proc_full_names = {'mortality':'All-cause mortality\n(Carleton et al. 2022)',
                           'gdppc':'GDP per capita growth\n(Burke, Hsiang, and\nMiguel 2015)',
                           'payroll':'Payroll\n(Behrer et al., 2021)'}
        
        
        unc_names = {'uncint':'Internal\nvariability',
                     'uncmod':'Model\nuncertainty',
                     'uncobs':'Reanalysis\nuncertainty'}
        
        uncert_typ = {'mortality':'13runs8mods',
                      'gdppc':'13runs8mods',
                      'payroll':'10runs7mods'}
        
        proj = {proc:ccrs.EckertIV() for proc in params_proc}
        proj['payroll'] = ccrs.AlbersEqualArea(central_longitude=-100,central_latitude=35)
        
        if typ == 'raw':
            cbar_params = {'mortality':{'vmin':0,'vmax':500,'cmap':cmocean.cm.amp},
                           'gdppc':{'vmin':0,'vmax':10,'cmap':cmocean.cm.amp},
                           'payroll':{'vmin':0,'vmax':0.00005,'cmap':cmocean.cm.amp}}
            clabel = 'Variance by uncertainty source'
        elif typ == 'norm':
            cbar_params = {proc:{'vmin':0,'vmax':1,'cmap':cmocean.cm.amp}
                           for proc in params_proc}
        clabel = 'Fraction of total variance by uncertainty source'
        
        plot_gwls = [1.5,2,3,4]
        
        
        fig = plt.figure(figsize=(18,12))
        gs = plt.GridSpec(nrows=3,
                          ncols=4*len(imp_params),
                          hspace=-0.05,wspace=0.2)
        
        # Now, just the map
        for col_idx,proc in enumerate(['mortality','gdppc','payroll']):
            #-------------------------
            # Load plot data ---------
            #-------------------------
            # Load shapefile
            gdf = load_shp(proc)
            
            # Load partitioned uncertainty
            fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}.zarr'
            uncerts = xr.open_zarr(fn).load()
            uncerts = uncerts.set_index(poly_idx = 'locid').rename({'poly_idx':'locid'})
        
            # Select desired specification
            uncerts = uncerts.sel(imp_params[proc])
            uncerts = uncerts.sel(typ=typ,sample=uncert_typ[proc],locid=gdf.locid.values,gwl=gwl)
        
            #-------------------------
            # Plot data --------------
            #-------------------------
            for row_idx,uncsrc in enumerate(['uncobs','uncint','uncmod']):
                if params_proc[proc]['geo_typ'] == 'adm2-us':
                    # Only use part of the horizontal grid spec for the smaller counties map
                    ax_map = fig.add_subplot(gs[row_idx,(col_idx*4):(col_idx*4+3)],projection=proj[proc])
                else: 
                    ax_map = fig.add_subplot(gs[row_idx,(col_idx*4):(col_idx*4+4)],projection=proj[proc])
        
                fig,ax_map=plot_map(uncerts[uncsrc],gdf,ax=ax_map,fig=fig,**cbar_params[proc])
                # Coastlines/plot edges
                ax_map.coastlines()
                gdf.dissolve().plot(facecolor='none',edgecolor='k',linewidth=0.5,ax=ax_map,transform=ccrs.PlateCarree())
    
                # Make the payroll figure span a little wider, so the aspect ratio fits better
                # with the global maps
                if proc == 'payroll':
                    ax_map.set_extent((-140, -64, 23.7, 50.6),
                              crs = ccrs.PlateCarree())
        
                # Column labels
                if col_idx == 0:
                    ax_map.annotate(unc_names[uncsrc],
                                (-0.1,0.5),xycoords = 'axes fraction',
                                fontweight='bold',fontsize=13,va='center',ha='center',
                                rotation=90,font=extra_fonts['lato-bold'])
        
                # Titles
                if row_idx == 0:
                    ax_map.set_title(proc_full_names[proc],
                                     font=extra_fonts['lato-bold'],
                                     fontsize=13)
        
                # Colorbar
                if row_idx == len(params_proc)-1:
                    cax = fig.add_axes([ax_map.get_position().x0+0.025,
                        ax_map.get_position().y0-0.05,
                        ax_map.get_position().width-0.05,
                        0.02])
                    # Vertical colorbar
                    norm = mpl.colors.Normalize(vmin=cbar_params[proc]['vmin'],vmax=cbar_params[proc]['vmax'])
                    sm = plt.cm.ScalarMappable(cmap=cbar_params[proc]['cmap'],norm=norm)
                    plt.colorbar(sm,cax=cax,label=clabel,orientation='horizontal')
        
                ax_map.annotate(string.ascii_lowercase[row_idx*3+col_idx],(0.01,1.01),xycoords = 'axes fraction',
                            fontweight='bold',fontsize=12,va='bottom',font=extra_fonts['lato-bold'])
        plt.tight_layout()
    
        if save_fig:
            utility_print(output_fn)
                


/glade/derecho/scratch/schwarzwald/tmp/ipykernel_42879/2947259263.py:109: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


/glade/u/home/schwarzwald/projects/impact_uncertainty/figures/extra_figs/uncert_source_maps_gwl1.5_raw.pdf saved!
/glade/u/home/schwarzwald/projects/impact_uncertainty/figures/extra_figs/uncert_source_maps_gwl1.5_raw.png saved!
